# 01 Build Loan-Month Panel

**Purpose:** Join origination traits to the monthly history and flag first default / first prepayment.

**Steps**
- Join on Loan Sequence Number
- Derive vintage quarter and MOB
- Default flag (90+ DPD / RA / loss ZBC), absorbing
- Prepay flag (ZBC 01)
- Loan-level first event table
- Sanity checks

**Read first:** Bellini 2.2 (default definition)

**Done when:** panel and loan-level event table for 2007 saved to `data/processed/`, checks pass.

**Year in this pass:** 2007 only. Functions that work get moved into `src/freddie/`.

## Setup

In [1]:
# Imports + Notebook Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.append(r"C:\Dev\CreditRiskLearning\Projects\Freddie Mac End-to-End Project\src")
from freddie.config import RAW_DIR, INTERIM_DIR

from freddie.load import load_orig, load_perf, sanity_check

years = [2006, 2007, 2010, 2016]

In [2]:
orig = {year: load_orig(year, save=False) for year in years}
orig[2007].head()

2006 orig: 50,000 rows x 31 cols
2007 orig: 50,000 rows x 31 cols
2010 orig: 50,000 rows x 31 cols
2016 orig: 50,000 rows x 31 cols


,credit_score,first_payment_date,first_time_homebuyer,maturity_date,msa,mi_pct,num_units,occupancy_status,orig_cltv,orig_dti,...,orig_loan_term,num_borrowers,seller_name,super_conforming_flag,pre_harp_loan_id,special_eligibility_program,harp_flag,property_valuation_method,interest_only_flag,vantage_score
0,769,200702,N,203701,24220,0,1,P,90,38,...,360,2,OTHER,N,<NA>,NaN,N,7,N,9999
1,598,200703,N,203702,<NA>,0,1,P,90,45,...,360,2,OTHER,N,<NA>,NaN,N,7,N,9999
2,777,200703,N,203702,20260,0,1,P,75,23,...,360,2,OTHER,N,<NA>,NaN,N,7,N,9999
3,628,200703,N,203702,35644,0,1,P,65,38,...,360,2,OTHER,N,<NA>,NaN,N,7,N,9999
4,696,200703,N,203702,<NA>,0,1,P,80,23,...,360,2,OTHER,N,<NA>,NaN,N,7,N,9999


In [3]:
perf= {year: load_perf(year, save=False) for year in years}
perf[2007].head()

,loan_id,period,current_upb,current_dlq_status,loan_age,zero_balance_code,zero_balance_date
0,F07Q10000023,200701,195000.0,00,0,NaN,<NA>
1,F07Q10000023,200702,195000.0,00,1,NaN,<NA>
2,F07Q10000023,200703,195000.0,00,2,NaN,<NA>
3,F07Q10000023,200704,195000.0,00,3,NaN,<NA>
4,F07Q10000023,200705,194000.0,00,4,NaN,<NA>


In [4]:
perf[2010].sample(10)

,loan_id,period,current_upb,current_dlq_status,loan_age,zero_balance_code,zero_balance_date
237579,F10Q10104775,201803,124187.66,00,97,NaN,<NA>
1779273,F10Q30108476,202212,62831.21,00,147,NaN,<NA>
680730,F10Q10305787,201112,71321.65,00,20,NaN,<NA>
819537,F10Q20000909,201402,70708.13,00,45,NaN,<NA>
870416,F10Q20023303,202502,255445.61,00,177,NaN,<NA>
896155,F10Q20035495,201605,63084.00,00,72,NaN,<NA>
1910962,F10Q30182865,201009,67000.00,00,0,NaN,<NA>
3044887,F10Q40351790,201409,158277.26,00,44,NaN,<NA>
2162702,F10Q30328544,201108,336490.32,00,10,NaN,<NA>
390666,F10Q10173080,201003,344000.00,00,0,NaN,<NA>


In [5]:
df_2006 = perf[2006]

delinquent = df_2006[df_2006['current_dlq_status'] != '00']

delinquent.sample(5)





,loan_id,period,current_upb,current_dlq_status,loan_age,zero_balance_code,zero_balance_date
1657259,F06Q30009989,201508,154468.30,36,108,NaN,<NA>
1726595,F06Q30049767,201703,28958.98,02,127,NaN,<NA>
2036437,F06Q30218397,201009,70356.20,26,48,NaN,<NA>
82424,F06Q10033123,200603,153000.00,01,1,NaN,<NA>
1460457,F06Q20314619,200704,148996.08,01,9,NaN,<NA>


In [6]:
# Join on Loan Sequence Number

In [7]:
# Add Vintage Quarter

def vintage_quarter(df):
    
    df['vintage'] = df['loan_id'].str.replace(r'^F(\d{2})(Q[1-4]).*', r'20\1\2', regex=True)
    
    return df

vintage_quarter(df_2006)

,loan_id,period,current_upb,current_dlq_status,loan_age,zero_balance_code,zero_balance_date,vintage
0,F06Q10000059,200602,180000.00,00,0,NaN,<NA>,2006Q1
1,F06Q10000059,200603,180000.00,00,1,NaN,<NA>,2006Q1
2,F06Q10000059,200604,180000.00,00,2,NaN,<NA>,2006Q1
3,F06Q10000059,200605,180000.00,00,3,NaN,<NA>,2006Q1
4,F06Q10000059,200606,179000.00,00,4,NaN,<NA>,2006Q1
...,...,...,...,...,...,...,...,...
3203494,F06Q40433166,202105,50199.89,00,173,NaN,<NA>,2006Q4
3203495,F06Q40433166,202106,50000.74,00,174,NaN,<NA>,2006Q4
3203496,F06Q40433166,202107,49800.55,00,175,NaN,<NA>,2006Q4
3203497,F06Q40433166,202108,49599.32,00,176,NaN,<NA>,2006Q4


In [8]:
# Add MOB

In [25]:
# Add 90+ DPD Flag

# DLQ status is string - contains non number values
# Need  >= '03' status code or 'RA'
# Loss ZBCs for your default flag: 02, 03, 09, 15

def default_flagger(df):

    loss_codes = ['02', '03', '09', '15']
    
    status_str = df["current_dlq_status"].astype(str)
    
    is_numeric_gte_03 = status_str.str.isdigit() & (
    status_str >= "03"
)
    is_ra = status_str == "RA"

    is_zbc_loss = df['zero_balance_code'].isin(loss_codes)

    
    df['default_flag'] = np.select(condlist=[is_numeric_gte_03 | is_ra | is_zbc_loss], choicelist=[1], default=0)
    
    return df

In [22]:
df_2006['zero_balance_code'].value_counts().head(60)

zero_balance_code
01    43643
09     2274
03     1013
16      968
02      547
15      235
96      227
Name: count, dtype: int64

In [26]:
df_2006.groupby('current_dlq_status')['default_flag'].mean()

current_dlq_status
00    0.000015
01    0.000034
02    0.000756
03    1.000000
04    1.000000
        ...   
95    1.000000
96    1.000000
97    1.000000
98    1.000000
99    1.000000
Name: default_flag, Length: 101, dtype: float64

In [10]:
# Add ZBD Flag

In [11]:
# Add Prepay ZBC01 

In [12]:
# Loan Level First Event Table